In [8]:
import copy
import math
import multiprocessing
import os
import random
import time
import warnings
from collections.abc import Callable
from functools import partial

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib_inline.backend_inline import set_matplotlib_formats
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torch.utils.flop_counter import FlopCounterMode

from dataset_loading import get_batch, prepare

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# mixed precision (float16) training on CUDA GPUs
USE_AMP = device.type == "cuda"

# channels-last memory layout speeds up convolutions on CUDA GPUs
MEMORY_FORMAT = (
    torch.channels_last if device.type == "cuda" else torch.contiguous_format
)

# pick the fastest convolution algorithms for our shapes
torch.backends.cudnn.benchmark = True

print(
    f"PyTorch {torch.__version__}, device: {device}"
)

PyTorch 2.14.1+cu130, device: cuda


## Training loop

In [9]:
from train import evaluate, fit, generate

## Model: LSTM

In [10]:
class LSTMLM(nn.Module):
    """LSTM language model."""

    def __init__(self, vocab_size: int, d_model: int, hidden: int, n_layers: int):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model) # -> (B, T, d_model)
        self.rnn = nn.LSTM(d_model, hidden, n_layers, batch_first=True) # -> (B, T, hidden) - returns hidden state of last layer

        self.proj = nn.Linear(hidden, d_model, bias=False) # -> (B, T, d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False) # -> (B, T, vocab_size)
        self.head.weight = self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight, std=0.02)

    def forward(self, idx):
        # idx: (B, T)
        out, _ = self.rnn(self.tok_emb(idx))
        return self.head(self.proj(out))

## Build the model

In [11]:
BLOCK_SIZE, BATCH_SIZE = 256, 64
D_MODEL, HIDDEN, N_LAYERS = 384, 880, 2

MODEL_PATH = "./data/lstm.pt"

tok = prepare()
model = LSTMLM(tok.get_vocab_size(), D_MODEL, HIDDEN, N_LAYERS).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    print(f"Loaded weights from {MODEL_PATH}")
else:
    print(f"No checkpoint at {MODEL_PATH}, starting from random init")

model.train()

17.3M parameters
Loaded weights from ./data/lstm.pt


LSTMLM(
  (tok_emb): Embedding(16384, 384)
  (rnn): LSTM(384, 880, num_layers=2, batch_first=True)
  (proj): Linear(in_features=880, out_features=384, bias=False)
  (head): Linear(in_features=384, out_features=16384, bias=False)
)

## Train

In [12]:
# history = fit(model, steps=50000, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, save_path=MODEL_PATH)

In [13]:
# plt.plot(history["step"], history["train_loss"], label="train (fineweb-edu)")
# plt.plot(history["step"], history["val_loss"], label="val (fineweb-edu)")
# plt.xlabel("step"), plt.ylabel("cross-entropy"), plt.legend();

## Test on Wikipedia

In [14]:
# test_loss = evaluate(model, "test", BLOCK_SIZE, BATCH_SIZE, n_batches=100)
# print(f"wikipedia test loss {test_loss:.3f}, perplexity {math.exp(test_loss):.1f}")

## Inference: prompt completions

In [15]:
prompts = [
    "The capital of France is",
    "In 1969, the first humans",
    "Photosynthesis is the process by which",
    "The most important thing in a good education is",
    "To make a good cup of coffee, you should",
    "The history of the Roman Empire began",
    "Scientists have recently discovered that",
    "The main difference between a virus and a bacterium is",
    "When learning a new language, it helps to",
    "The president announced yesterday that",
]
for p in prompts:
    print(f"PROMPT: {p}\nOUTPUT: {generate(model, tok, p, max_new_tokens=60, block_size=BLOCK_SIZE)}\n")

PROMPT: The capital of France is
OUTPUT: The capital of France is a major building of power, with a central focus on the history of the world. The first attempt to build the power to capitalise in England and the Untonia. The most important role played by an elite, however, is the influence of the industrial revolution and the world. The influence of

PROMPT: In 1969, the first humans
OUTPUT: In 1969, the first humans had been established upon the site of the Czarkkawaris in December 2006.
Khao's discovery in the Django River's earliest origins in the 1960s, the Situ-Huku-Kango Szakuo, and

PROMPT: Photosynthesis is the process by which
OUTPUT: Photosynthesis is the process by which the chemical reactions begin to form a chemical reaction that occurs when the cells are under constant pressure.
This process is called the ‘Functional Revolution’ in Pelvic as a process of producing enzymes (in the process of growth) and then applying the same chemical reaction to the reaction process.

PR

## Inference time

In [16]:
# inference speed: generate a fixed number of tokens (stop_at_eot=False so the length is exact)
N_NEW = 200
generate(model, tok, prompts[0], 10, block_size=BLOCK_SIZE, stop_at_eot=False)  # warm-up
if device.type == "cuda":
    torch.cuda.synchronize()

start = time.time()
generate(model, tok, prompts[0], N_NEW, block_size=BLOCK_SIZE, stop_at_eot=False)
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - start

print(f"{N_NEW} tokens in {elapsed:.2f} s -> {N_NEW / elapsed:.1f} tokens/s")

200 tokens in 7.99 s -> 25.0 tokens/s
